# ESdE Adultos 2023: Data audit and population-representation validation

## Introduction

The scope of this notebook is to provide exploratory analysis of the integrity of the public-use ESdE 2023 adult microdata, with focus on item-level missingness, questionaire routing patterns, and population characterization, for the intention of future explotarion of predictors of health outcomes in following notebooks. 

Its intention is not to reproduce INE's full evaluation of non-response and population representation, which can obtained [here](https://www.ine.es/metodologia/t15/esde23_falta_res.pdf), along the official INE's ESdE [methodology](https://www.ine.es/metodologia/t15/esde23_meto.pdf). 

In [ ]:
import pandas as pd
from ine_health_data.pipeline import load_variables, start_setup, is_nonresponse, get_codebook
start_setup() 

## Data audit and integrity

### General dataset shape

According to INEs methodology, the number of completed adult questionnaires was 21040 out of 21085 surveyed households.
However in the non-response report, 21077 households were included, with 45 total indicences of non-response.

In the dataset, the total number of rows should represent the number of surveyed adults, which can be contrasted with the household ID `IDENTHOGAR` due to its uniqueness as the sample selects for one adult 15+ whithin each household. 

In [ ]:
all_raw_data = load_variables()
pd.Series({
    "n rows": len(all_raw_data),
    "n cols": len(all_raw_data.columns),
    "n IDENTHOGAR": all_raw_data["IDENTHOGAR"].count(),
    "n NaN IDENTHOGAR": all_raw_data["IDENTHOGAR"].isna().sum(),
    "max IDENTHOGAR ID": all_raw_data["IDENTHOGAR"].max(),
    "duplicated IDENTHOGAR": all_raw_data["IDENTHOGAR"].duplicated().sum(),
}).to_frame(name="value")

The dataset available contains 21032 rows, indicating fewer completed adult questionnaires than reported by methodology document. 
However, based on the non-response document, the count is explained by the 45 incidences of the 21077 household reported, resulting in a total of 21032 surveyed adults.

The discrepancy of 8 records from both INE's official documents is not explained within any of both, and will be ignored due to being out of the scope of this project. 


The amount of columns loaded from the full dataset are in accordance to the number of variables described in the codebook adjacent to the dataset raw file.

### Missingness, item non-response, and questionnaire routing 

As the questionnaire contains conditional response fields that depend on previous answers, it is important to differentiate between routed-out questions, which should be represented with a missing/blank value, and "non-response" or "non-applicable" represented with special code responses.

The table below provides a general overview of these response patterns in the dataset in total ~~and per variable group~~ #TODO.

In [ ]:
total_cells = all_raw_data.size
codebook = get_codebook()

total_na = all_raw_data.isna().sum().sum()
total_no_contesta = is_nonresponse(df=all_raw_data,nona_labels="No contesta",codebook=codebook).sum().sum()
total_no_aplicable = is_nonresponse(df=all_raw_data,nona_labels="No aplicable",codebook=codebook).sum().sum()
total_no_consta = is_nonresponse(df=all_raw_data,nona_labels="No consta",codebook=codebook).sum().sum()

count = pd.Series({
    "Total cells":total_cells,
    "Blank":total_na,
    "'No contesta'": total_no_contesta,
    "'No aplicable'": total_no_aplicable,
    "'No consta'": total_no_consta
})
fract = pd.Series({
    "Total cells": 1,
    "Blank":total_na/total_cells,
    "'No contesta'": total_no_contesta/total_cells,
    "'No aplicable'": (total_no_aplicable/total_cells),
    "'No consta'": (total_no_consta/total_cells)
})
summary = pd.concat({
        "n":count,
        "fract":fract
    },axis=1
).sort_values(by="fract",ascending=False)
display(summary)
total_non_full = total_na+total_no_contesta+total_no_aplicable+total_no_consta
display(f"Total Non-response={total_non_full}; {(total_non_full/total_cells).round(4)}")